# Replicating Cont, Kukanov and Stoikov (2011): the price impact of order book events

Fedya's seminar notebook for the paper's baseline: **Tables 2, 3 and 4 and Figures 5 and 6**,
estimated on two datasets, with the formulas, the hypothesis tests and a verdict table computed
from the results. All estimation code lives in the tested modules of `research/baseline/` (OLS
per window, HC3 and wild-bootstrap inference, the depth law, intraday seasonality); this notebook
calls them and presents what they return. The code written here is presentation (tables, one
prototype figure), three small diagnostics of the quadratic term (Section 5), the rule that
flags degenerate fits (Section 4), and the window-count tests and verdict rules of Section 9.
That logic is new and lives only in this notebook; it is not covered by the repository's unit
tests.

**Contents**

1. Data and how to run
2. Setup and loading
3. Model and formulas
4. Estimation (every model, in one cell)
5. Table 2: linear impact and the quadratic term
6. Table 4: order flow imbalance against trade imbalance
7. Table 3 and a Figure 4 prototype: impact against depth
8. Figures 5 and 6: intraday seasonality on BX
9. Hypotheses and verdicts
10. Next step for Timur: does deeper OFI add out-of-sample R²?
11. Limitations

Paper: R. Cont, A. Kukanov, S. Stoikov, *The price impact of order book events*,
[arXiv:1011.6402](https://arxiv.org/abs/1011.6402) (Journal of Financial Econometrics, 2014).

## 1. Data and how to run

| | Small sample | Final dataset |
| --- | --- | --- |
| Source | NASDAQ order book via LOBSTER, five levels | NASDAQ BX ITCH 5.0, five levels |
| Stocks and dates | AAPL, MSFT on 2012-06-21 | 10 stocks on 10 dates, 2018-12-28 to 2020-03-02 |
| Feature files | `outputs/features/10s` (computed by the Rust engine, README "Quick start" 2-3) | `data/order_book_final/features/10s` from release `dataset-2026-10-15` ([guide](../../docs/FINAL_DATASET.md)) |
| Caveats | one day: 13 half-hour windows per stock | trade imbalance unknown in 19,498 of 111,456 rows (hidden executions without a side); ADBE, GOOG and QCOM are sparse; one venue, not the NBBO |

Neither dataset is in Git. Section 2 checks that both exist and stops with download or build
instructions if not.

**Units and conventions.** $\Delta P$ is the mid-price change in ticks of one cent
(`mid_price_change_ticks`); OFI, trade imbalance and depth are in shares. Buckets are 10 seconds,
`(start, end]`. Windows are half-hours on the :00/:30 grid (a UTC floor, which coincides with
New York :00/:30). Buckets rejected by the engine's quality checks are missing, never zero.

**How to run.** From the repository root run `uv sync --locked --all-extras` once. Open this file
in VS Code (Jupyter extension) or JupyterLab and select the project interpreter `.venv`. If VS
Code offers to install `ipykernel`, accept: it is installed into `.venv` only (`uv pip install
ipykernel` does the same, and a later `uv sync` may remove it again). Then run all cells from the
top. The notebook finds the repository root itself, writes no files and uses no IPython magics,
so its cells also run in a plain Python interpreter. Section 4 prints the estimation time; the
whole notebook, including the 999-draw bootstrap on BX, ran in about 15 seconds when it was
written.

## 2. Setup and loading

The first cell walks up from the working directory to the folder holding `pyproject.toml` and
puts it on `sys.path`, so that `research.*` imports work when Jupyter starts in
`research/notebooks/`. `ofi_pipeline` is the installed package and needs no path.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "research" / "baseline").is_dir():
    raise RuntimeError(
        "Start the notebook inside the order-book-impact repository: no folder with "
        "pyproject.toml and research/baseline was found above the working directory."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repository root:", ROOT)

In [ ]:
from dataclasses import replace
import json
import time
import warnings

import matplotlib
from matplotlib import pyplot
from matplotlib.figure import Figure
import numpy as np
import pandas as pd
from scipy.stats import binom

from ofi_pipeline.research import load_features
from research.baseline.depth_impact import DepthImpactConfig, paper_table3, window_depth_betas
from research.baseline.run import BaselineConfig, fit_models
from research.baseline.seasonality import (
    AXIS, BLUE, GRID, INK, INK_2, MUTED, ORANGE, SURFACE, SeasonalityConfig,
    cross_stock_average, figure5, figure6, sample_label, seasonal_summary, stock_patterns,
    window_statistics,
)  # fmt: skip
from research.baseline.tables import paper_table2, paper_table4

# The figures below are plain matplotlib Figure objects (from research.baseline.seasonality and
# from this notebook), not pyplot figures. matplotlib loads its backend lazily, and only the
# Jupyter inline backend, once loaded, registers how a Figure is displayed; without this call
# a cell ending with a figure shows the text "<Figure size ...>" instead of the image (checked
# with ipykernel 7.2 and matplotlib 3.10.7). In a plain interpreter the call is harmless.
pyplot.switch_backend(matplotlib.get_backend())
# Two statsmodels messages from degenerate windows (Section 4): the HC0 Wald F with a singular
# covariance (the tables use the classical F) and HC3's division by 1 - leverage when the
# leverage is one (fit_models then reports HC3 and the bootstrap as undefined).
warnings.filterwarnings(
    "ignore", message="covariance of constraints|divide by zero encountered", module="statsmodels"
)
pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.0f}" if 1000 <= abs(x) < 1e6 else f"{x:.3g}",
)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)  # the verdict table of Section 9 holds short texts

LEVEL = 0.05  # significance level of every test in the notebook
INFERENCES = ["hc0", "hc3", "bootstrap"]  # the paper's z-test on HC0, then the robust ones
NEW_YORK = "America/New_York"

In [ ]:
SMALL_DIR = ROOT / "outputs" / "features" / "10s"
BX_ROOT = ROOT / "data" / "order_book_final"
BX_DIR = BX_ROOT / "features" / "10s"
GET_SMALL = (
    "small LOBSTER sample: unpack Lev's bundle into data/order_book_sample/ and run "
    "`python scripts/prepare_features.py --sample-dir data/order_book_sample "
    "--output-dir outputs/features/10s` (README.md, Quick start 2-3)"
)
GET_BX = (
    "final NASDAQ BX dataset: download order_book_final_models.zip from the GitHub release "
    "dataset-2026-10-15 and unzip it into data/ (docs/FINAL_DATASET.md, section Download)"
)
REQUIRED_FILES = {
    SMALL_DIR / "features.parquet": GET_SMALL,
    BX_DIR / "features.parquet": GET_BX,
    BX_ROOT / "reports" / "table1.csv": GET_BX,
    BX_ROOT / "data" / "features_10s.parquet": GET_BX,
}
missing = [
    f"- {path.relative_to(ROOT)}: {hint}"
    for path, hint in REQUIRED_FILES.items()
    if not path.is_file()
]
if missing:
    raise FileNotFoundError("Input data are missing (they are not in Git):\n" + "\n".join(missing))

small = load_features(SMALL_DIR)
bx = load_features(BX_DIR)
DATASETS = {"LOBSTER": small, "BX": bx}
for name, data in DATASETS.items():
    print(f"{name}: {len(data):,} accepted 10-second buckets, {data['symbol'].nunique()} stocks, "
          f"{data['date'].nunique()} dates")  # fmt: skip

Per stock: dates, accepted buckets, the share of buckets with unknown trade imbalance, the mean
absolute price change, the mean L1 depth and, for BX, the mean quoted spread from the dataset's
own descriptive table (`reports/table1.csv`). Stocks with a mean BX spread below 5 cents are
called **narrow-spread** below; they are the closest analogue of the paper's mostly one-tick
stocks.

In [ ]:
spread = pd.read_csv(BX_ROOT / "reports" / "table1.csv", index_col="symbol")["mean_spread_cents"]
NARROW = sorted(spread.index[spread < 5])
print("narrow-spread BX stocks (mean spread below 5 cents):", ", ".join(NARROW))

overview = (
    pd.concat(DATASETS, names=["dataset"])
    .groupby(["dataset", "symbol"], sort=False)
    .agg(
        dates=("date", "nunique"),
        buckets=("ofi_l1", "size"),
        ti_unknown_pct=("trade_imbalance", lambda values: 100 * values.isna().mean()),
        mean_abs_dp_ticks=("mid_price_change_ticks", lambda values: values.abs().mean()),
        mean_l1_depth=("avg_depth", "mean"),
    )
)
overview["bx_mean_spread_cents"] = [
    spread.get(symbol, np.nan) if dataset == "BX" else np.nan for dataset, symbol in overview.index
]
overview

## 3. Model and formulas

**Order flow imbalance.** At every update $n$ of the best quotes (bid price $P^B_n$, bid size
$q^B_n$, ask price $P^A_n$, ask size $q^A_n$) the paper records the event's contribution

$$e_n = \mathbf{1}\{P^B_n \ge P^B_{n-1}\}\,q^B_n - \mathbf{1}\{P^B_n \le P^B_{n-1}\}\,q^B_{n-1}
      - \mathbf{1}\{P^A_n \le P^A_{n-1}\}\,q^A_n + \mathbf{1}\{P^A_n \ge P^A_{n-1}\}\,q^A_{n-1}$$

and sums it over the bucket $[t_{k-1}, t_k]$, where $N(t)$ counts the events up to $t$:

$$\mathrm{OFI}_k = \sum_{n=N(t_{k-1})+1}^{N(t_k)} e_n .$$

The mid-price change in ticks is $\Delta P_k = (P_k - P_{k-1})/\delta$ with $\delta$ = 1 cent.
The Rust engine computes both (`ofi_l1`, `mid_price_change_ticks`); this notebook never
recomputes them.

**Linear impact and depth.** If every price level beyond the best quotes holds $D$ shares, the
paper's stylised book gives $\Delta P = \mathrm{OFI}/(2D) + \epsilon$ (eq. 1). In general

$$\Delta P_k = \beta\,\mathrm{OFI}_k + \epsilon_k \tag{2}$$

$$\beta_i = \frac{c}{AD_i^{\lambda}} + \nu_i \tag{3}$$

where $AD_i$ is the average depth of half-hour window $i$: the mean of $(q^B_n + q^A_n)/2$ over
its quote updates. The stylised book corresponds to $\lambda = 1$ and $c = 1/2$. Implementation
(`window_depth_betas`): each bucket's `avg_depth` is the mean of $(q^B + q^A)/2$ over its L1
updates, and the window averages the buckets with weights `quote_update_count`.

**Per-window regressions (Table 2).** For every stock and window $i$ (about 180 buckets $k$),
by OLS with an intercept:

$$\Delta P_k = \hat\alpha_i + \hat\beta_i\,\mathrm{OFI}_k + \hat\epsilon_k \tag{4}$$

$$\Delta P_k = \hat\alpha_{Q,i} + \hat\beta_{Q,i}\,\mathrm{OFI}_k
               + \hat\gamma_{Q,i}\,\mathrm{OFI}_k\,|\mathrm{OFI}_k| + \hat\epsilon_{Q,k}$$

Table 2 averages $\hat\alpha_i$, $\hat\beta_i$ and $R^2$ of (4), $\hat\gamma_{Q,i}$ of the
second regression and their z-statistics over windows, and reports the share of windows where
each coefficient is significant at 5%.

**Depth law (Table 3, Figure 4).** Per stock, across its windows, with Newey-West standard
errors and normal confidence intervals:

$$\log\hat\beta_i = \hat\alpha_{L} - \hat\lambda\,\log AD_i + \hat\epsilon_{L,i} \tag{5}$$

$$\hat\beta_i = \hat\alpha_{M} + \hat c\,AD_i^{-\hat\lambda} + \hat\epsilon_{M,i} \tag{6}$$

Windows with $\hat\beta_i \le 0$ have no logarithm; they are dropped and counted.

**Variance decomposition (Figures 5 and 6).** Taking variances of (2) within window $i$:

$$\operatorname{var}[\Delta P_k]_i = \beta_i^2\,\operatorname{var}[\mathrm{OFI}_k]_i
  + \operatorname{var}[\epsilon_k]_i \tag{7}$$

so the intraday pattern of $\operatorname{var}[\Delta P]$ should follow that of
$\beta^2\operatorname{var}[\mathrm{OFI}]$.

**Trades (Table 4).** With the trade imbalance $TI_k = \sum_n b_n - \sum_n s_n$ (buyer- minus
seller-initiated volume in the bucket):

$$\Delta P_k = \hat\alpha_i + \hat\beta_i\,\mathrm{OFI}_k + \hat\epsilon_k \tag{8a}$$

$$\Delta P_k = \hat\alpha_{T,i} + \hat\beta_{T,i}\,TI_k + \hat\epsilon_{T,k} \tag{8b}$$

$$\Delta P_k = \hat\alpha_{D,i} + \hat\theta_{O,i}\,\mathrm{OFI}_k + \hat\theta_{T,i}\,TI_k
               + \hat\epsilon_{D,k} \tag{8c}$$

If OFI already contains the effect of trades, $\hat\theta_{T,i}$ in (8c) is indistinguishable
from zero.

**Inference.** The paper uses a two-sided z-test on White (HC0) standard errors. With design
matrix $X$, residuals $\hat\epsilon_k$ and leverages $h_k = [X(X^\top X)^{-1}X^\top]_{kk}$:

$$\widehat V_{\mathrm{HC0}} = (X^\top X)^{-1} X^\top \operatorname{diag}(\hat\epsilon_k^2)\,
  X (X^\top X)^{-1}, \qquad
  \widehat V_{\mathrm{HC3}}:\ \hat\epsilon_k^2 \ \to\ \hat\epsilon_k^2 / (1-h_k)^2 .$$

HC0 standard errors are too small when a few buckets carry high leverage, which is what a
heavy-tailed regressor such as $\mathrm{OFI}\,|\mathrm{OFI}|$ produces in a window of about 180
buckets; HC3 inflates exactly those residuals. The **wild bootstrap-t** (`research/baseline/inference.py`)
imposes the null $\theta_j = 0$, refits the restricted model ($\tilde y$, $\tilde\epsilon$),
draws $y^*_k = \tilde y_k + w_k\,\tilde\epsilon_k$ with Rademacher weights $w_k = \pm 1$,
recomputes the HC3 t-statistic $t^*$ and reports
$p = \bigl(1 + \#\{|t^*| \ge |t|\}\bigr) / (1 + B)$. Both are undefined (missing) in a window
where one bucket has leverage one, i.e. the fit passes exactly through it. In 400 simulated
windows with a true linear model and t(3)-distributed regressors, the nominal 5% test rejected
34.5% of the time with HC0, 10% with HC3 and 11% with the bootstrap (CHANGELOG, step 1C).
Tables 2 and 4 are shown under all three; **the verdicts in Section 9 use the bootstrap.**
Table 3 uses Newey-West errors across windows, as in the paper.

## 4. Estimation

Every model is estimated once, here, with the repository's configuration files
(`configs/baseline.json`, `configs/depth_impact.json`, `configs/seasonality.json`): target
`mid_price_change_ticks`, 30-minute windows, at least 20 buckets per window, a wild bootstrap
with the configured number of draws on **both** datasets.

* **Table 2**: regression (4) and its quadratic extension on every accepted bucket (trade
  imbalance is not needed).
* **Table 4**: regressions (8a-c) only on buckets whose trade signs are all known
  (`trade_signing_complete`), so that the three models of a window share the same buckets. On
  the small sample this is every bucket; on BX it drops the buckets with unknown trade
  imbalance.
* **Table 3 and Figures 5-6**: $\hat\beta_i$ from an OFI-only fit on every bucket with a known
  price change and OFI, and $AD_i$ per window.

The last table counts, per model, the windows that were fitted (`ok`) and those skipped
(too few buckets, constant target, or a rank-deficient design).

In [ ]:
def read_config(name: str) -> dict:
    return json.loads((ROOT / "configs" / f"{name}.json").read_text())


BASELINE = BaselineConfig(**read_config("baseline"))
TABLE2_CONFIG = replace(
    BASELINE, models={"ofi_l1": ["ofi_l1"], "ofi_quadratic": ["ofi_l1", "ofi_l1_signed_square"]}
)
TABLE4_CONFIG = replace(
    BASELINE,
    models={
        "ofi_l1": ["ofi_l1"],
        "trade_imbalance": ["trade_imbalance"],
        "ofi_and_ti": ["ofi_l1", "trade_imbalance"],
    },
)
DEPTH_CONFIG = DepthImpactConfig(**read_config("depth_impact"))
SEASONALITY_CONFIG = SeasonalityConfig(**read_config("seasonality"))

started = time.perf_counter()
fits, depth_windows = {}, {}
for name, data in DATASETS.items():
    fits[name, "table2"] = fit_models(data, TABLE2_CONFIG)
    fits[name, "table4"] = fit_models(data[data["trade_signing_complete"]], TABLE4_CONFIG)
    depth_windows[name] = window_depth_betas(data, DEPTH_CONFIG)
bx_window_stats = window_statistics(bx, SEASONALITY_CONFIG)
print(
    f"estimated in {time.perf_counter() - started:.0f} s; wild bootstrap with "
    f"{BASELINE.bootstrap_draws} draws, seed {BASELINE.bootstrap_seed}"
)

pd.concat(
    {
        key: metrics.groupby("model")["status"].value_counts().unstack(fill_value=0)
        for key, (metrics, _) in fits.items()
    },
    names=["dataset", "table", "model"],
).fillna(0).astype(int)

### Degenerate fits

In some windows a regressor is non-zero in only one or a few buckets: for example a single
bucket with any signed trade volume, or two or three buckets with any order flow while every
other bucket has $\mathrm{OFI} = 0$ and $\Delta P = 0$. The fit then passes (almost) exactly
through those buckets: their leverage approaches one, their residuals vanish, the HC0 standard
error collapses and the HC0 z-statistic can become enormous. When the leverage is numerically
one, HC3 and the bootstrap are undefined. Just below that they are defined but just as
meaningless (an HC3 z of exactly 1 is the signature of a coefficient fixed by a single
bucket), and the bootstrap can then call the coefficient significant. Such windows are not
confined to the sparse stocks: the list below also contains stocks with more than a hundred
fitted windows, and they affect HC3 and the bootstrap as well as HC0.

**Rule, a judgment call.** A fit is flagged when its HC3 statistic is undefined or its in-sample
$R^2$ exceeds 0.99. In Table 2 only the flagged model is dropped in that window, because the
linear and the quadratic regression answer separate questions. In Table 4 the whole window is
dropped from all three models, which must share the same buckets. Dropped fits count as
skipped (status `degenerate`). Tables 2 and 4, their significance comparisons and the
window-based verdicts (H1, H2, H4a, H4b) use these cleaned fits; the raw paper-layout tables
follow as secondary tables in Sections 5 and 6, and the verdict table of Section 9 repeats
each window-based verdict on the raw fits, so the effect of the rule on the conclusions is
visible. Table 3, Figures 5-6 and therefore H3 and H5 keep every window, as their modules do.
The 0.99 threshold is arbitrary: the cell also counts the fits with $R^2$ between 0.98 and 0.99,
which are kept. The list gives each flagged window with the models that triggered the flag, the
largest $R^2$ and the largest HC0 |z| among its coefficients.

In [ ]:
R2_LIMIT = 0.99
WINDOW = ["symbol", "date", "window_start"]
JOINT = {"table2": False, "table4": True}  # Table 4's three models must share their windows
flags, clean_fits = [], {}
for (name, table), (metrics, coefficients) in fits.items():
    by_fit = coefficients.groupby([*WINDOW, "model"])
    fitted = (
        metrics[metrics["status"] == "ok"]
        .set_index([*WINDOW, "model"])
        .assign(
            hc3_undefined=by_fit["hc3_p_value"].apply(lambda p: p.isna().any()),
            largest_abs_z_hc0=by_fit["z_statistic"].apply(lambda z: z.abs().max()),
        )
        .reset_index()
    )
    suspect = fitted[fitted["hc3_undefined"] | (fitted["in_sample_r2"].astype(float) > R2_LIMIT)]
    flags.append(suspect.assign(dataset=name, table=table))
    keys = WINDOW if JOINT[table] else [*WINDOW, "model"]  # drop the window, or only the fit
    flagged = pd.MultiIndex.from_frame(suspect[keys])
    dropped = pd.MultiIndex.from_frame(metrics[keys]).isin(flagged) & metrics["status"].eq("ok")
    clean_fits[name, table] = (
        metrics.assign(status=metrics["status"].mask(dropped, "degenerate")),
        coefficients[~pd.MultiIndex.from_frame(coefficients[keys]).isin(flagged)],
    )
    r2 = fitted["in_sample_r2"].astype(float)
    print(
        f"{name} {table}: {len(suspect[WINDOW].drop_duplicates())} of "
        f"{len(fitted[WINDOW].drop_duplicates())} fitted windows flagged, {int(dropped.sum())} of "
        f"{len(fitted)} fits dropped; kept fits with R2 in (0.98, 0.99]: "
        f"{int(((r2 > 0.98) & (r2 <= R2_LIMIT)).sum())}"
    )
flags = pd.concat(flags, ignore_index=True)
flags["slot"] = flags["window_start"].dt.tz_convert(NEW_YORK).dt.strftime("%H:%M")
flags.groupby(["dataset", "table", "symbol", "date", "slot"]).agg(
    models=("model", ", ".join),
    buckets=("observations", "max"),
    max_r2=("in_sample_r2", "max"),
    hc3_undefined=("hc3_undefined", "any"),
    largest_abs_z_hc0=("largest_abs_z_hc0", "max"),
)

## 5. Table 2: linear impact and the quadratic term

Paper layout under the paper's own test (HC0 z), on the cleaned fits of Section 4. `windows`
counts the windows used, `t_*` are mean z-statistics over windows, `pct_*_sig` the percentage
of windows significant at 5% and `r2` the mean in-sample $R^2$ of (4). The last row is the
paper's average over 50 stocks (April 2010, TAQ, NBBO) for reference. $\beta$ is in ticks per
share, so its size depends on depth and price and is not comparable across stocks or samples.

In [ ]:
PAPER = "paper (2010 TAQ)"


def stack(tables: dict, paper_row: dict | None = None) -> pd.DataFrame:
    # Stack per-dataset paper-layout tables (index: dataset, symbol) and the paper's average.
    stacked = pd.concat(
        {name: table.set_index("symbol") for name, table in tables.items()}, names=["dataset"]
    )
    if paper_row is None:
        return stacked
    index = pd.MultiIndex.from_tuples([(PAPER, "Average")], names=["dataset", "symbol"])
    return pd.concat([stacked, pd.DataFrame([paper_row], index=index)])


def by_inference(tables: dict, measures: list[str]) -> pd.DataFrame:
    # Selected columns of one paper table under each inference, side by side.
    wide = pd.concat(
        {
            kind: stack({name: tables[name, kind] for name in DATASETS})[measures]
            for kind in INFERENCES
        },
        axis=1,
    )
    wide.columns = wide.columns.swaplevel()
    # The bootstrap table reports the HC3 mean z, so its t-columns repeat the HC3 ones.
    return wide[
        [
            (m, k)
            for m in measures
            for k in INFERENCES
            if not (m.startswith(("t_", "both_t_")) and k == "bootstrap")
        ]
    ]


# table2: every fitted window (as the CLI writes it); table2_clean: degenerate windows excluded.
table2, table2_clean = {}, {}
for name in DATASETS:
    for kind in INFERENCES:
        table2[name, kind] = paper_table2(*fits[name, "table2"], inference=kind)
        table2_clean[name, kind] = paper_table2(*clean_fits[name, "table2"], inference=kind)
PAPER_TABLE2 = {
    "windows": 273, "alpha": 0.0002, "t_alpha": -0.02, "beta": 0.0398, "t_beta": 11.47,
    "gamma_q": -2.0e-4, "t_gamma_q": -0.28, "r2": 0.65, "pct_alpha_sig": 6.0,
    "pct_beta_sig": 97.0, "pct_gamma_q_sig": 9.0,
}  # fmt: skip
stack({name: table2_clean[name, "hc0"] for name in DATASETS}, PAPER_TABLE2)

### The same significance shares under HC3 and the wild bootstrap

Why change the test: HC0 standard errors are too small when a window's fit leans on a few
high-leverage buckets, and $\mathrm{OFI}\,|\mathrm{OFI}|$ is dominated by its largest values, so
the HC0 z-test of $\gamma_Q$ rejects a true null far more often than 5% (the simulation in
Section 3). The table puts the three tests side by side on the cleaned fits; the mean
z-statistics are shown for HC0 and HC3 (the bootstrap reuses the HC3 statistic).

In [ ]:
table2_significance = by_inference(table2_clean, ["pct_beta_sig", "pct_gamma_q_sig", "t_gamma_q"])
for name in DATASETS:
    share = table2_significance.loc[(name, "Average"), "pct_gamma_q_sig"]
    print(
        f"{name}: gamma_Q significant in {share['hc0']:.0f}% of windows (stock average) under HC0, "
        f"{share['hc3']:.0f}% under HC3 and {share['bootstrap']:.0f}% under the bootstrap"
    )
table2_significance

*Secondary: the raw Table 2 on every fitted window*, as `python -m research.baseline.run`
writes it (`table2.csv`). Here the degenerate windows of Section 4 dominate some HC0 means: the
mean z of $\gamma_Q$ reaches astronomical values for a few BX stocks and for the BX Average.

In [ ]:
stack({name: table2[name, "hc0"] for name in DATASETS})

### Why the quadratic term looked significant on the small sample

Under the paper's HC0 test the term $\mathrm{OFI}\,|\mathrm{OFI}|$ looked significant in most
AAPL windows and in many MSFT windows of the small sample (CHANGELOG, step 1B), against 9% in
the paper. Three findings, each checked by a cell below:

1. **Inference.** HC0 over-rejects here (Section 3). With HC3 or the bootstrap the share of
   significant windows roughly halves on both datasets (the comparison table above). It does
   not fall to the paper's level everywhere; Section 9 tests whether what remains is more
   than chance.
2. **MSFT: one bucket, and the thin opening book.** The per-window table shows where the
   significance sits. The window with the most extreme HC0 z owes it to a single bucket:
   dropping that bucket and refitting the window removes the effect, and without that window
   the mean HC0 z shrinks markedly (second cell). Few MSFT windows stay significant under the
   bootstrap; the cell prints them with their depth relative to the median window, and the
   opening window, which has the thinnest book of the day, is among them. The investigation
   behind CHANGELOG step 1C (untracked files in `outputs/baseline/quadratic_investigation/`)
   also found that letting impact depend on depth removes most of the remaining MSFT
   curvature; that analysis is not reproduced here.
3. **AAPL: concave, but not quadratic.** The quadratic coefficient of AAPL is negative in
   every window (the z-statistics above), but the shape is not quadratic. The last cell
   compares, per stock, the linear model (4), the quadratic model and a signed power law
   $\Delta P_k = a_i + b_i\,\mathrm{sign}(\mathrm{OFI}_k)\,|\mathrm{OFI}_k|^p$ with
   window-specific $a_i, b_i$ and one exponent $p$ per stock chosen on a grid. For AAPL the
   best $p$ lies well below 1, and in-sample the quadratic term (one extra coefficient per
   window) removes about as much squared error as the power law (one extra parameter). On
   held-out data (each window split into five contiguous blocks; fit on four, score the fifth,
   with $p$ chosen on the four) the power law keeps a large gain for AAPL, while the quadratic
   model does worse than the linear one: $\mathrm{OFI}\,|\mathrm{OFI}|$ extrapolates badly to
   the largest imbalances. For MSFT the held-out gains of both are a few percent.

*Diagnostics written in this notebook*: the per-window table, the single-bucket refit (which
reuses `fit_models`) and the power-law comparison.

In [ ]:
_, small_coefficients = clean_fits["LOBSTER", "table2"]
small_gamma = small_coefficients[
    (small_coefficients["model"] == "ofi_quadratic")
    & (small_coefficients["term"] == "ofi_l1_signed_square")
].merge(
    depth_windows["LOBSTER"][
        ["symbol", "date", "window_start", "observations", "beta", "avg_depth"]
    ],
    on=["symbol", "date", "window_start"],
)
small_gamma["slot"] = small_gamma["window_start"].dt.tz_convert(NEW_YORK).dt.strftime("%H:%M")
per_window = small_gamma.pivot(
    index="slot",
    columns="symbol",
    values=[
        "observations",
        "avg_depth",
        "beta",
        "z_statistic",
        "hc3_z_statistic",
        "bootstrap_p_value",
    ],
).rename(
    columns={
        "observations": "n",
        "avg_depth": "AD",
        "z_statistic": "z_gamma_hc0",
        "hc3_z_statistic": "z_gamma_hc3",
        "bootstrap_p_value": "p_gamma_boot",
    }
)
per_window.columns = per_window.columns.swaplevel()
per_window.sort_index(axis=1, level=0, sort_remaining=False).rename_axis("slot (New York)")

In [ ]:
msft = small_gamma[small_gamma["symbol"] == "MSFT"].set_index("window_start")
worst = msft["z_statistic"].idxmin()
window = small[(small["symbol"] == "MSFT") & (small["interval_start"].dt.floor("30min") == worst)]
largest = window["ofi_l1"].abs().idxmax()
print(
    f"MSFT {msft.loc[worst, 'slot']} window: the largest |OFI| is {window.loc[largest, 'ofi_l1']:,.0f} "
    f"shares, {abs(window.loc[largest, 'ofi_l1']) / window['ofi_l1'].std():.1f} standard deviations "
    f"of the window's OFI"
)
print(
    f"MSFT HC0 z of gamma over windows: mean {msft['z_statistic'].mean():.2f}, "
    f"median {msft['z_statistic'].median():.2f}; mean without this window "
    f"{msft['z_statistic'].drop(worst).mean():.2f}"
)
significant = msft[msft["bootstrap_p_value"] < LEVEL]
print(
    "MSFT windows significant under the bootstrap: "
    + ", ".join(
        f"{row.slot} (AD {row.avg_depth / msft['avg_depth'].median():.2f}x the median window)"
        for row in significant.itertuples()
    )
)
refits = {
    "all buckets": window,
    "without the largest-|OFI| bucket": window.drop(index=largest),
}
pd.concat(
    {
        label: fit_models(rows, TABLE2_CONFIG)[1]
        .set_index(["model", "term"])
        .loc[
            ("ofi_quadratic", "ofi_l1_signed_square"),
            ["estimate", "z_statistic", "hc3_z_statistic", "bootstrap_p_value"],
        ]
        for label, rows in refits.items()
    },
    axis=1,
).T.rename_axis(f"MSFT {msft.loc[worst, 'slot']} window")

In [ ]:
POWERS = [float(p) for p in np.round(np.arange(0.20, 1.51, 0.05), 2)]  # 1.0 is model (4)
MODELS = ["quadratic", *POWERS]
FOLDS = 5  # contiguous blocks of every window for the held-out comparison


def regressors(model, x: np.ndarray) -> np.ndarray:
    columns = [x, x * np.abs(x)] if model == "quadratic" else [np.sign(x) * np.abs(x) ** model]
    return np.column_stack([np.ones_like(x), *columns])


def sse(model, x_fit, y_fit, x_eval, y_eval) -> float:
    # Squared error on (x_eval, y_eval) of an OLS fit (with intercept) on (x_fit, y_fit).
    coefficients = np.linalg.lstsq(regressors(model, x_fit), y_fit, rcond=None)[0]
    residual = y_eval - regressors(model, x_eval) @ coefficients
    return float(residual @ residual)


# Squared errors summed over a stock's windows: training fit per (stock, fold, model), where
# fold -1 is the fit on all buckets, and held-out error per (stock, fold, model).
train, test = {}, {}
for (symbol, _), rows in small.groupby(["symbol", small["interval_start"].dt.floor("30min")]):
    y = rows["mid_price_change_ticks"].to_numpy(dtype=float)
    x = rows["ofi_l1"].to_numpy(dtype=float)
    block = np.arange(len(y)) * FOLDS // len(y)  # buckets are in time order
    for model in MODELS:
        for fold in range(-1, FOLDS):
            fit, held = block != fold, block == fold
            key = (symbol, fold, model)
            train[key] = train.get(key, 0.0) + sse(model, x[fit], y[fit], x[fit], y[fit])
            if fold >= 0:
                test[key] = test.get(key, 0.0) + sse(model, x[fit], y[fit], x[held], y[held])

shape = {}
for symbol in sorted(small["symbol"].unique()):
    # The exponent is chosen on the training blocks of each fold, then scored on the held-out one.
    chosen = [POWERS[np.argmin([train[symbol, f, p] for p in POWERS])] for f in range(FOLDS)]
    held_out = {
        "linear": sum(test[symbol, f, 1.0] for f in range(FOLDS)),
        "quadratic": sum(test[symbol, f, "quadratic"] for f in range(FOLDS)),
        "power": sum(test[symbol, f, p] for f, p in zip(range(FOLDS), chosen)),
    }
    in_sample = [train[symbol, -1, p] for p in POWERS]
    shape[symbol] = {
        "best_power": POWERS[np.argmin(in_sample)],
        "in_sample_gain_power_pct": 100 * (1 - min(in_sample) / train[symbol, -1, 1.0]),
        "in_sample_gain_quadratic_pct": 100
        * (1 - train[symbol, -1, "quadratic"] / train[symbol, -1, 1.0]),
        "held_out_gain_power_pct": 100 * (1 - held_out["power"] / held_out["linear"]),
        "held_out_gain_quadratic_pct": 100 * (1 - held_out["quadratic"] / held_out["linear"]),
        "powers_chosen_in_folds": ", ".join(f"{p:.2f}" for p in chosen),
    }
pd.DataFrame.from_dict(shape, orient="index").rename_axis("gain = squared-error reduction vs (4)")

## 6. Table 4: order flow imbalance against trade imbalance

Panel A of the paper's Table 4 in its layout (HC0), on the cleaned fits of Section 4. `ofi_*` is
regression (8a), `ti_*` (8b), `both_*` (8c); `*_f` is the mean classical F statistic. On BX the
three models are fitted on the **same buckets, those with complete trade signs**, so their
$R^2$ and statistics are comparable; the BX `ofi_*` columns therefore differ slightly from
Table 2, which uses every bucket. The last row is the paper's grand mean.

In [ ]:
# table4: every fitted window (as the CLI writes it); table4_clean: degenerate windows excluded.
table4, table4_clean = {}, {}
for name in DATASETS:
    for kind in INFERENCES:
        table4[name, kind] = paper_table4(*fits[name, "table4"], inference=kind)
        table4_clean[name, kind] = paper_table4(*clean_fits[name, "table4"], inference=kind)
for name, data in DATASETS.items():
    known = int(data["trade_signing_complete"].sum())
    print(
        f"{name}: {known:,} of {len(data):,} buckets ({100 * known / len(data):.1f}%) "
        "have complete trade signs"
    )
PAPER_TABLE4 = {
    "windows": 273, "ofi_r2": 0.65, "ofi_t_beta": 11.47, "ofi_pct_beta_sig": 97.0, "ofi_f": 429.0,
    "ti_r2": 0.32, "ti_t_beta": 4.18, "ti_pct_beta_sig": 81.0, "ti_f": 103.0, "both_r2": 0.67,
    "both_t_theta_ofi": 8.49, "both_t_theta_ti": 1.16, "both_pct_theta_ofi_sig": 95.0,
    "both_pct_theta_ti_sig": 31.0, "both_f": 231.0,
}  # fmt: skip
stack({name: table4_clean[name, "hc0"] for name in DATASETS}, PAPER_TABLE4)

The question of Table 4 is whether $\theta_T$ in (8c) is zero once OFI is in the regression.
Its significance share under the three tests (with trade imbalance alone for comparison), on
the cleaned fits:

In [ ]:
by_inference(table4_clean, ["ti_pct_beta_sig", "both_pct_theta_ti_sig", "both_t_theta_ti"])

*Secondary: the raw Table 4 on every fitted window*, as the CLI writes it (`table4.csv`). The
degenerate windows of Section 4 inflate some HC0 mean z and F values here; an infinite F comes
from a window fitted exactly.

In [ ]:
stack({name: table4[name, "hc0"] for name in DATASETS})

## 7. Table 3 and a Figure 4 prototype: impact against depth

Equations (5) and (6) per stock (`paper_table3`). Columns: estimates with Newey-West t-statistics
and 95% intervals; fit measures `r2_loglog` (the $R^2$ of eq. 5), `corr2_fitted` (squared
correlation of $\hat\beta_i$ with $\hat c\,AD_i^{-\hat\lambda}$) and `corr2_inverse_depth`
(with $1/AD_i$, the $\lambda = 1$ prediction); `p_lambda_eq_1` tests $\lambda = 1$, and the
Average of `lambda_eq_1_not_rejected` is the share of stocks where it is not rejected (paper:
35 of 50). The small sample has only 13 windows per stock. Where $\hat\lambda$ is near zero the
regressor of (6) is nearly constant and $\hat c$ is not identified, so it should not be read
(this also makes the BX Average of `c` meaningless). Compare `bx_spread_cents` with
`lambda_hat` and `r2_loglog`: on BX the depth law fits only where the spread is narrow, as the
printed ranges show. The paper also reports poor fits for its few wide-spread, thin stocks. The
BX `Average` row therefore mixes two regimes and is not comparable with the paper's grand mean;
the `narrow-spread mean` and `wide-spread mean` rows (plain means over each group's stocks) are
the better comparison. Table 3 keeps every fitted window, as `paper_table3` does, including the
few degenerate OFI-only windows of Section 4.

In [ ]:
table3 = {name: paper_table3(windows, DEPTH_CONFIG) for name, windows in depth_windows.items()}
PAPER_TABLE3 = {
    "windows": 273, "c": 0.45, "lambda_hat": 0.98, "t_c": 20.74, "t_lambda": 29.53, "c_low": 0.38,
    "c_high": 0.52, "lambda_low": 0.88, "lambda_high": 1.08, "r2_loglog": 0.74,
    "corr2_fitted": 0.75, "corr2_inverse_depth": 0.75, "lambda_eq_1_not_rejected": 35 / 50,
}  # fmt: skip
bx_table3 = table3["BX"][table3["BX"]["symbol"] != "Average"].set_index("symbol")
spread_groups = {"narrow": NARROW, "wide": sorted(set(bx_table3.index) - set(NARROW))}
# The BX Average mixes two regimes, so add the mean of each spread group next to it.
group_rows = pd.DataFrame(
    {
        f"{group}-spread mean": bx_table3.loc[symbols].mean()
        for group, symbols in spread_groups.items()
    }
).T
bx_view = pd.concat([table3["BX"].set_index("symbol"), group_rows]).rename_axis("symbol")
table3_view = stack({"LOBSTER": table3["LOBSTER"], "BX": bx_view.reset_index()}, PAPER_TABLE3)
spread_lookup = spread.to_dict() | {
    f"{group}-spread mean": spread[symbols].mean() for group, symbols in spread_groups.items()
}
table3_view.insert(
    0,
    "bx_spread_cents",
    [
        spread_lookup.get(symbol, np.nan) if dataset == "BX" else np.nan
        for dataset, symbol in table3_view.index
    ],
)
for group, symbols in spread_groups.items():
    members = bx_table3.loc[symbols]
    print(
        f"BX {group}-spread stocks ({', '.join(symbols)}): lambda-hat "
        f"{members['lambda_hat'].min():.2f} to {members['lambda_hat'].max():.2f}, R2 of eq. (5) "
        f"{members['r2_loglog'].min():.2f} to {members['r2_loglog'].max():.2f}"
    )
table3_view

**Figure 4 prototype.** The paper illustrates eq. (5) with a log-log scatter of $\hat\beta_i$
against $AD_i$ for one stock. Here, on BX, one narrow-spread stock (AMD, which is also in the
paper's 2010 sample, with $\hat\lambda = 0.94$ and $R^2$ = 78% there) and one wide-spread stock
(AAPL); one point per half-hour window with $\hat\beta_i > 0$, and the eq. (5) least-squares
line, whose slope is $-\hat\lambda$ of Table 3 (the cell checks this). Table 3 has every stock.

In [ ]:
FIGURE4_STOCKS = {"AMD": BLUE, "AAPL": ORANGE}
bx_depth = depth_windows["BX"]
figure4 = Figure(figsize=(10, 4.6), facecolor=SURFACE, layout="constrained")
for ax, (symbol, color) in zip(figure4.subplots(1, 2), FIGURE4_STOCKS.items()):
    points = bx_depth[
        (bx_depth["symbol"] == symbol) & (bx_depth["status"] == "ok") & (bx_depth["beta"] > 0)
    ]
    log_depth, log_beta = np.log(points["avg_depth"]), np.log(points["beta"])
    slope, intercept = np.polyfit(log_depth, log_beta, 1)
    assert np.isclose(-slope, bx_table3.loc[symbol, "lambda_hat"])  # the line is eq. (5)
    ends = np.array([log_depth.min(), log_depth.max()])
    ax.scatter(log_depth, log_beta, s=36, color=color, edgecolors=SURFACE, linewidths=1.5, zorder=3)
    ax.plot(ends, intercept + slope * ends, color=INK_2, linewidth=2, zorder=2)
    ax.set_title(
        f"{symbol}: mean spread {spread[symbol]:.1f} cents, {len(points)} windows\n"
        f"λ̂ = {-slope:.2f}, R² of eq. (5) = {bx_table3.loc[symbol, 'r2_loglog']:.2f}",
        loc="left", fontsize=10, color=INK,
    )  # fmt: skip
    ax.set_xlabel("log AD (average L1 depth, shares)", fontsize=9, color=INK_2)
    ax.set_ylabel("log β (ticks per share)", fontsize=9, color=INK_2)
    ax.set_facecolor(SURFACE)
    ax.grid(color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, labelsize=9, length=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(AXIS)
figure4.suptitle(
    "Impact against depth per half-hour window, NASDAQ BX (paper Figure 4 prototype)",
    x=0.01, ha="left", fontsize=12, fontweight="semibold", color=INK,
)  # fmt: skip
figure4

## 8. Figures 5 and 6: intraday seasonality on BX

Per stock and half-hour slot of the New York trading day, $\hat\beta_i$, $AD_i$,
$\operatorname{var}[\Delta P]$ and $\operatorname{var}[\mathrm{OFI}]$ are averaged over days and
divided by the stock's own average over all windows; the figures show the mean across stocks
(`research/baseline/seasonality.py`). A slot needs at least `min_days` windows for a stock and
a stock needs `min_slots` valid slots. $\beta^2 \operatorname{var}[\mathrm{OFI}]$ is built from
the average patterns of its factors (as in the paper's footnote 4) and scaled like
$\operatorname{var}[\Delta P]$. A stock with fewer than `min_days` windows in a slot has no
value there, so the cross-stock mean of that slot covers fewer stocks: the cell prints the
stocks without a 09:30 slot, and the 09:30 ratios below compare a slightly smaller set of
stocks than the other slots (the figure subtitles give the stock count per slot). The small
sample has one day, so one window per slot: no pattern can be averaged and it is not used here.

Two groups: every BX stock with enough coverage, and the narrow-spread stocks. The table gives
the numbers the paper quotes (`seasonal_summary`): the 09:30 value divided by the mean over
slots, the 09:30 value over the 15:30 value, the peak slots and the correlation of the two
curves of Figure 6 across slots. The labels inside Figure 5 instead divide by the stock's
average over all windows, so they can differ slightly from the table. The paper column is read
from its Section 3.3 (depth "two times lower" and impact "two times higher" at the open than on
average, impact "five times higher" at the open than at the close, var[ΔP] peaking near the
open and var[OFI] near the close). These patterns are descriptive; no test is attached.

In [ ]:
seasonal = {}
groups = {"all eligible": sorted(bx_window_stats["symbol"].unique()), "narrow spread": NARROW}
for group, symbols in groups.items():
    stats = bx_window_stats[bx_window_stats["symbol"].isin(symbols)]
    patterns, excluded = stock_patterns(stats, SEASONALITY_CONFIG)
    average = cross_stock_average(patterns)
    seasonal[group] = {
        "patterns": patterns,
        "average": average,
        "summary": seasonal_summary(average),
        "label": sample_label(stats, patterns, average, "NASDAQ BX"),
    }
    opening = patterns[(patterns["slot"] == "09:30") & patterns["beta"].notna()]
    no_open = sorted(set(patterns["symbol"]) - set(opening["symbol"]))
    print(
        f"{group}: {', '.join(sorted(patterns['symbol'].unique()))}; excluded: {excluded or 'none'}; "
        f"without a 09:30 slot (fewer than {SEASONALITY_CONFIG.min_days} opening windows): "
        f"{', '.join(no_open) or 'none'}"
    )

PAPER_SEASONALITY = {
    "beta_open_over_average": "about 2",
    "depth_open_over_average": "about 0.5",
    "beta_open_over_close": "about 5",
    "var_dp_peak_slot": "near the open",
    "var_ofi_peak_slot": "near the close",
}
seasonal_view = pd.DataFrame(
    {
        group: {
            key: round(value, 2) if isinstance(value, float) else value
            for key, value in parts["summary"].items()
        }
        for group, parts in seasonal.items()
    }
)
seasonal_view[PAPER] = pd.Series(PAPER_SEASONALITY)
seasonal_view

In [ ]:
figure5(seasonal["all eligible"]["average"], seasonal["all eligible"]["label"])

In [ ]:
figure6(seasonal["all eligible"]["average"], seasonal["all eligible"]["label"])

In [ ]:
figure5(seasonal["narrow spread"]["average"], seasonal["narrow spread"]["label"])

In [ ]:
figure6(seasonal["narrow spread"]["average"], seasonal["narrow spread"]["label"])

## 9. Hypotheses and verdicts

The rules are fixed here and applied per stock: H1, H2, H4a and H4b to the cleaned fits of
Tables 2 and 4, H3 and H5 to every window (Table 3 and the seasonal patterns). Window-level
tests use the wild bootstrap at 5%. The verdict table repeats the window-based verdicts on the
raw fits (`verdict_raw_fits`), so the effect of the degenerate-fit rule is visible. Two kinds
of statement need different wording:

* **Claims** (H1, H4a, H5) state a property; the verdict says for how many stocks it *holds*.
* **Null hypotheses** (H2, H3, H4b) can only be *rejected* or *not rejected*, and a
  non-rejection is not evidence for the null when the test has little power. A non-rejection is
  therefore reported as *inconclusive* when the stock has fewer than 30 windows (H2, H4b) or
  when the 95% interval of $\lambda$ includes 0 (H3: too imprecise to tell $\lambda = 1$ from no
  depth effect at all). The power table after the count tests shows what the 30-window cut-off
  means.

| | Hypothesis (paper) | Kind | Test | Outcome per stock |
| --- | --- | --- | --- | --- |
| H1 | Linear impact: $\beta > 0$ in eq. (4) | claim | per-window test of $\beta = 0$ | holds if $\beta$ is significant in at least 90% of its windows and its mean $\hat\beta > 0$ |
| H2 | No quadratic term: $\gamma_Q = 0$ | null | per-window test of $\gamma_Q = 0$, then a one-sided binomial test of the number of significant windows against a null share of 11% | rejected if the binomial p is below 5%; otherwise not rejected, or inconclusive with fewer than 30 windows |
| H3 | $\beta \propto 1/AD$: $\lambda = 1$ in eq. (3) | null | Newey-West z-test of $\lambda = 1$ in eq. (5) | rejected if p < 5%; otherwise not rejected if the interval of $\lambda$ lies above 0, else inconclusive |
| H4a | OFI explains more than trades | claim | mean $R^2$ of (8a) against (8b) | holds if $R^2_{\mathrm{OFI}} > R^2_{TI}$ |
| H4b | Trades add nothing given OFI: $\theta_T = 0$ in (8c) | null | as H2, for $\theta_T$ | as H2 |
| H5 | Impact is highest when the book is thin, at the open | claim, descriptive | intraday patterns, no test | holds if the normalised $\beta$ at 09:30 is above 1 and the normalised $AD$ below 1; a stock without a 09:30 slot is not evaluated |

**The null share of 11% is a heuristic.** Even when the null is true, the bootstrap flags some
windows: in the simulation of CHANGELOG step 1C its rejection rate at the nominal 5% was 11%.
That comes from a single design (t(3) regressors, 400 windows, so about ±1.6 points), and the
binomial test also treats a stock's windows as independent. The first table gives, per stock,
the windows, the significant ones, the smallest count that would reject, the binomial p-value
and the outcome at 11%, plus the outcome at the nominal 5% as a sensitivity check. The
verdict table repeats both.

In [ ]:
NULL_SHARE = 0.11  # the bootstrap's simulated rejection rate under a true null (heuristic)
MIN_WINDOWS = 30  # below this a non-rejection is reported as inconclusive (low power)


def count_outcome(k: int, n: int, share: float) -> str:
    # Binomial test of k significant windows out of n against the null share.
    if binom.sf(k - 1, n, share) < LEVEL:  # P(at least k significant windows)
        return "rejected"
    return "inconclusive" if n < MIN_WINDOWS else "not rejected"


def window_test(coefficients: pd.DataFrame, model: str, term: str) -> pd.DataFrame:
    # Per stock: windows with a bootstrap p-value, the significant ones and the count test.
    chosen = coefficients[(coefficients["model"] == model) & (coefficients["term"] == term)]
    p_values = chosen.dropna(subset=["bootstrap_p_value"]).groupby("symbol")["bootstrap_p_value"]
    result = pd.DataFrame(
        {"windows": p_values.size(), "significant": p_values.agg(lambda p: int((p < LEVEL).sum()))}
    )
    n, k = result["windows"], result["significant"]
    result["share_pct"] = 100 * k / n
    result["min_to_reject"] = [
        next((j for j in range(m + 1) if binom.sf(j - 1, m, NULL_SHARE) < LEVEL), np.nan) for m in n
    ]
    result["binomial_p"] = binom.sf(k - 1, n, NULL_SHARE)
    result["outcome"] = [count_outcome(a, b, NULL_SHARE) for a, b in zip(k, n)]
    result["outcome_null_5pct"] = [count_outcome(a, b, 0.05) for a, b in zip(k, n)]
    return result


TESTED_TERMS = {
    "H2 gamma_Q": ("table2", "ofi_quadratic", "ofi_l1_signed_square"),
    "H4b theta_T": ("table4", "ofi_and_ti", "trade_imbalance"),
}
window_tests = pd.concat(
    {
        (label, name): window_test(clean_fits[name, table][1], model, term)
        for label, (table, model, term) in TESTED_TERMS.items()
        for name in DATASETS
    },
    names=["test", "sample", "symbol"],
)
window_tests

**Power of the count test.** For a stock with a given number of windows: the smallest number of
significant windows that rejects the 11% null share, and the probability of rejecting when the
true share of significant windows is 20%, 25% or 30%. With few windows a real excess is often
missed: at a true share of 25% the test rejects in about 42% of cases with 13 windows (the small
sample) and in about 65% with 30 windows. That is why a non-rejection with fewer than 30
windows is reported as inconclusive rather than as support for the null; 30 is a convention,
not a sharp boundary.

In [ ]:
power = {}
for n in (13, 30, 42, 100):
    k = next(j for j in range(n + 1) if binom.sf(j - 1, n, NULL_SHARE) < LEVEL)
    power[n] = {"min_to_reject": k} | {
        f"power_true_{round(100 * share)}pct": binom.sf(k - 1, n, share)
        for share in (0.20, 0.25, 0.30)
    }
pd.DataFrame.from_dict(power, orient="index").rename_axis("windows")

In [ ]:
def holds(passed: pd.Series) -> str:
    # Verdict of a claim checked per stock.
    k, n = int(passed.sum()), len(passed)
    if n == 0:
        return "no data"
    return f"holds for all {n} stocks" if k == n else f"holds for {k} of {n} stocks"


def tally(outcomes: pd.Series) -> str:
    # Verdict of a null hypothesis tested per stock, e.g. "rejected 8, inconclusive 2 (of 10)".
    if outcomes.empty:
        return "no data"
    counts = outcomes.value_counts()
    found = [label for label in ("rejected", "not rejected", "inconclusive") if label in counts]
    return ", ".join(f"{label} {counts[label]}" for label in found) + f" (of {len(outcomes)})"


def listing(index) -> str:
    return ", ".join(index) or "none"


def per_stock(table: pd.DataFrame) -> pd.DataFrame:
    return table[table["symbol"] != "Average"].set_index("symbol")


def window_verdicts(source: dict, name: str) -> dict:
    # The window-based verdicts (H1, H2, H4a, H4b) on one set of fits, cleaned or raw.
    boot2 = per_stock(paper_table2(*source[name, "table2"], inference="bootstrap"))
    hc0_4 = per_stock(paper_table4(*source[name, "table4"]))
    tests = {
        label: window_test(source[name, table][1], model, term)
        for label, (table, model, term) in TESTED_TERMS.items()
    }
    return {
        "H1": holds((boot2["pct_beta_sig"] >= 90) & (boot2["beta"] > 0)),
        "H2": tally(tests["H2 gamma_Q"]["outcome"]),
        "H4a": holds(hc0_4["ofi_r2"] > hc0_4["ti_r2"]),
        "H4b": tally(tests["H4b theta_T"]["outcome"]),
    }


tests_by_sample = window_tests.sort_index()  # sorted for fast, warning-free lookups
rows = []
for name in DATASETS:
    raw = window_verdicts(fits, name)
    boot2, hc0_2 = per_stock(table2_clean[name, "bootstrap"]), per_stock(table2_clean[name, "hc0"])
    h1 = (boot2["pct_beta_sig"] >= 90) & (boot2["beta"] > 0)
    rows.append({
        "hypothesis": "H1 linear impact", "sample": name,
        "result": f"beta significant in {boot2['pct_beta_sig'].mean():.0f}% of windows "
                  f"(HC0 {hc0_2['pct_beta_sig'].mean():.0f}%); R2 {hc0_2['r2'].mean():.2f}",
        "paper": "97% (HC0); R2 0.65",
        "verdict": holds(h1),
        "detail": f"fails: {listing(h1.index[~h1])}",
        "verdict_raw_fits": raw["H1"],
    })  # fmt: skip

    tests = tests_by_sample.loc[("H2 gamma_Q", name)]
    rows.append({
        "hypothesis": "H2 no quadratic term", "sample": name,
        "result": f"gamma_Q significant in {boot2['pct_gamma_q_sig'].mean():.0f}% of windows "
                  f"(HC0 {hc0_2['pct_gamma_q_sig'].mean():.0f}%)",
        "paper": "9% (HC0)",
        "verdict": tally(tests["outcome"]),
        "detail": f"rejected: {listing(tests.index[tests['outcome'] == 'rejected'])}",
        "verdict_null_5pct": tally(tests["outcome_null_5pct"]),
        "verdict_raw_fits": raw["H2"],
    })  # fmt: skip

    t3 = per_stock(table3[name])
    t3 = t3[t3["lambda_hat"].notna()]
    h3 = pd.Series(
        np.where(t3["p_lambda_eq_1"] < LEVEL, "rejected",
                 np.where(t3["lambda_low"] > 0, "not rejected", "inconclusive")),
        index=t3.index,
    )  # fmt: skip
    rows.append({
        "hypothesis": "H3 lambda = 1", "sample": name,
        "result": f"lambda-hat from {t3['lambda_hat'].min():.2f} to {t3['lambda_hat'].max():.2f}",
        "paper": "0.98 on average; not rejected for 35 of 50",
        "verdict": tally(h3),
        "detail": f"not rejected: {listing(h3.index[h3 == 'not rejected'])}; "
                  f"inconclusive: {listing(h3.index[h3 == 'inconclusive'])}",
        "verdict_raw_fits": "(every window used)",
    })  # fmt: skip

    hc0_4, hc3_4 = per_stock(table4_clean[name, "hc0"]), per_stock(table4_clean[name, "hc3"])
    boot4 = per_stock(table4_clean[name, "bootstrap"])
    h4a = hc0_4["ofi_r2"] > hc0_4["ti_r2"]
    rows.append({
        "hypothesis": "H4a OFI explains more than TI", "sample": name,
        "result": f"mean R2: OFI {hc0_4['ofi_r2'].mean():.2f}, TI {hc0_4['ti_r2'].mean():.2f}, "
                  f"both {hc0_4['both_r2'].mean():.2f}",
        "paper": "0.65, 0.32, 0.67",
        "verdict": holds(h4a),
        "detail": f"fails: {listing(h4a.index[~h4a])}",
        "verdict_raw_fits": raw["H4a"],
    })  # fmt: skip

    tests = tests_by_sample.loc[("H4b theta_T", name)]
    rows.append({
        "hypothesis": "H4b TI adds nothing given OFI", "sample": name,
        "result": f"theta_T significant in {boot4['both_pct_theta_ti_sig'].mean():.0f}% "
                  f"(HC0 {hc0_4['both_pct_theta_ti_sig'].mean():.0f}%); mean z of TI alone -> next "
                  f"to OFI: HC0 {hc0_4['ti_t_beta'].mean():.2f} -> {hc0_4['both_t_theta_ti'].mean():.2f}, "
                  f"HC3 {hc3_4['ti_t_beta'].mean():.2f} -> {hc3_4['both_t_theta_ti'].mean():.2f}",
        "paper": "31% (HC0); z 4.18 -> 1.16 (HC0)",
        "verdict": tally(tests["outcome"]),
        "detail": f"rejected: {listing(tests.index[tests['outcome'] == 'rejected'])}",
        "verdict_null_5pct": tally(tests["outcome_null_5pct"]),
        "verdict_raw_fits": raw["H4b"],
    })  # fmt: skip

for group, parts in seasonal.items():
    patterns = parts["patterns"]
    opening = patterns[patterns["slot"] == "09:30"].set_index("symbol")
    no_open = sorted(set(patterns["symbol"]) - set(opening.dropna(subset=["beta", "depth"]).index))
    opening = opening.dropna(subset=["beta", "depth"])
    h5 = (opening["beta"] > 1) & (opening["depth"] < 1)
    summary = parts["summary"]
    rows.append({
        "hypothesis": "H5 impact highest at the thin open", "sample": f"BX, {group}",
        "result": f"09:30: beta {summary['beta_open_over_average']:.2f}x, AD "
                  f"{summary['depth_open_over_average']:.2f}x the slot mean; beta open/close "
                  f"{summary['beta_open_over_close']:.1f}",
        "paper": "about 2x and 0.5x; about 5",
        "verdict": f"{holds(h5)} (descriptive)",
        "detail": f"fails: {listing(h5.index[~h5])}; no 09:30 slot: {listing(no_open)}",
        "verdict_raw_fits": "(every window used)",
    })  # fmt: skip

verdicts = pd.DataFrame(rows).set_index(["hypothesis", "sample"]).fillna("")
verdicts

**Reading the table.**

* *Replicates on both samples:* OFI explains price changes far better than trade imbalance, in
  every stock (H4a). On the small sample the impact coefficient is also positive and
  significant in nearly every window (H1).
* *Replicates on BX, not clearly on the small sample:* "trade imbalance adds nothing once OFI is
  in the regression" (H4b) is not rejected for most BX stocks, and the mean z of trade
  imbalance falls by a factor of about four next to OFI, as in the paper. On the small sample
  it is rejected for AAPL and inconclusive for MSFT (13 windows); at the nominal 5% null share
  (`verdict_null_5pct`) it would be rejected for both. In every case the HC0 test overstates how often
  trade imbalance matters.
* *Replicates only for narrow-spread stocks:* on BX the impact coefficient clears the 90% bar of
  H1 for every narrow-spread stock but only for some wide-spread ones (`detail`). Impact falls
  with depth (H3 and Table 3: only they have a 95% interval of $\lambda$ above zero and an
  $R^2$ of eq. (5) above 0.3), but $\lambda = 1$ itself is not rejected for only part of them;
  on the small sample it is not rejected for either stock. The thin-book open with high impact
  (H5) holds for every narrow-spread stock with a 09:30 slot and for none of the wide-spread
  ones evaluated, with smaller magnitudes than in the paper. On wide-spread BX stocks $\beta$ is
  barely related to L1 depth.
* *Does not replicate:* the absence of a quadratic term (H2). Robust inference roughly halves
  the share of significant windows, but the null is still rejected for AAPL and for most BX
  stocks; the BX exceptions (one not rejected, one inconclusive) both have fewer than 50
  windows. On the small sample the concavity is concentrated in AAPL, whose impact curve is
  closer to a power law than to a quadratic (Section 5).

None of the window-based verdicts changes when the degenerate fits are kept
(`verdict_raw_fits`).

## 10. Next step for Timur: does deeper OFI add out-of-sample $R^2$?

**Hypothesis H6.** On the BX test dates, a regularised model of the contemporaneous price change
on the order flow imbalance of levels 1 to 5 (Ridge on the five OFI columns, or OLS on their
first principal components) has a higher out-of-sample $R^2$ than the level-1 baseline (4).

**Protocol and caveats.**

1. **Use the dataset's chronological split** (table below): fit on `train`, choose the penalty
   or the number of components on `validation`, report `test` once. No random splits: buckets
   of one session are dependent. GOOG has almost no validation or test buckets and QCOM
   almost no test buckets, so leave them out of per-stock test claims or report them apart.
2. **Scale on the training data only**, within stock (and, if scaling within window, with
   statistics of past windows only). OFI is in shares, and the price change per share of OFI
   ($\beta$, Table 2) differs by orders of magnitude across BX stocks, mostly with price and
   spread, while their mean L1 depth stays within a few hundred shares (Section 2). An
   unscaled pooled model mostly learns which stock it is looking at.
3. **Leakage.** `ofi_l2`...`ofi_l5` follow the *rank* of a level, not a fixed price. When the
   best quote is depleted, the old level 2 becomes level 1 and every deeper rank shifts, so deep
   OFI partly records the price move itself. Gains from L2-L5 can therefore be mechanical. At
   least report the gain separately for buckets in which the mid-price did not move; a
   fixed-price deep OFI, which would have to be added to the Rust engine, is the clean test.
4. **Compare against a nonlinear level-1 baseline too**, not only the linear one: on the small
   sample a concave level-1 power law already improves held-out fit for AAPL (Section 5). An
   L1-L5 gain should exceed what a flexible function of `ofi_l1` alone achieves.
5. **Test the difference properly.** Compare squared errors bucket by bucket on the test dates,
   with standard errors clustered by stock and session (or a block bootstrap); the HC0 z-test
   is too liberal here (Section 3).
6. This is a contemporaneous fit, not a forecast: OFI and $\Delta P$ describe the same interval.
   A forecasting claim needs a future target built with `lag_feature`.

In [ ]:
annotated = load_features(BX_ROOT / "data" / "features_10s.parquet")
split_counts = annotated.groupby(["symbol", "split"]).size().unstack("split")  # buckets
split_counts.loc["dates (any stock)"] = annotated.groupby("split")["date"].nunique()
split_counts[["train", "validation", "test"]].fillna(0).astype(int)

## 11. Limitations

* **Venue.** BX is a single NASDAQ venue; the paper uses the consolidated NBBO from TAQ. A
  single venue's best quotes are never tighter than the consolidated ones, and on BX several
  stocks have mean spreads of tens of cents or more (Section 2). Those wide-spread stocks behave
  unlike the paper's mostly one-tick stocks (Table 3 and Figure 4).
* **Sample size.** The small sample is one day of two stocks (13 windows each). BX has 10
  dates, not the paper's 21 consecutive days, and ADBE, GOOG and QCOM have few usable windows.
  The paper has 50 stocks.
* **Trade signs.** On BX, hidden executions have no side; buckets with an unknown sign are left
  out of Table 4 for all three models, which changes the sample relative to Table 2.
* **Inference.** Shares of significant windows move in steps of one window (about 8 points on
  the small sample). The binomial rule of Section 9 uses a heuristic null share and treats
  windows as independent. HC3 and the bootstrap are still somewhat liberal in simulation
  (Section 3). Fits flagged as degenerate (Section 4) are left out of Tables 2 and 4 and of
  the window-based verdicts, not of Table 3, the seasonal patterns, H3 or H5; the rule and its
  0.99 threshold are a judgment call, and the verdict table shows the raw-fit verdicts beside
  the cleaned ones.
* **Descriptive parts.** Figures 5 and 6 and the H5 verdict describe the sample; no test is
  attached, and a handful of stocks per slot (see the figure subtitles) is few.
* **Diagnostics.** The explanations of the quadratic term in Section 5 rest on one day of two
  stocks; the parts attributed to the earlier investigation are not reproduced here.
* **In-sample.** Every $R^2$ here is a contemporaneous in-sample fit, not out-of-sample
  accuracy (Section 10).